# Omni Voiceover Studio (Google Colab)

Turns your script into a professional voiceover, free on Colab's GPU.

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Click **▷ Run all**. When asked, **connect Google Drive** (your voices and voiceovers are kept in *My Drive → Omni*).
3. Wait 3–5 minutes, then open the `gradio.live` link at the bottom.
4. When finished: **Runtime → Disconnect and delete runtime** (saves your free GPU time).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU: switch Runtime to T4 GPU'

In [ ]:
# 1. Get the tool (or update it to the latest version)
%cd /content
!git clone -b claude/serene-edison-dhp0kf https://github.com/khleeq92-netizen/Omni.git 2>/dev/null || (cd /content/Omni && git pull)
%cd /content/Omni
!pip install -q -r requirements.txt

In [ ]:
# 2. Keep your voices, pronunciation fixes and voiceovers in Google Drive (My Drive > Omni)
#    When asked, click "Connect to Google Drive" and allow access.
#    Set USE_DRIVE = False only if you do not want anything saved between sessions.
USE_DRIVE = True

import os, shutil
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    base = '/content/drive/MyDrive/Omni'
    for sub in ['voices', 'output']:
        target = f'{base}/{sub}'
        os.makedirs(target, exist_ok=True)
        if os.path.islink(sub):
            os.unlink(sub)
        elif os.path.isdir(sub):  # keep anything made before Drive was connected
            for f in os.listdir(sub):
                src = os.path.join(sub, f)
                if os.path.isfile(src) and not os.path.exists(os.path.join(target, f)):
                    shutil.copy2(src, target)
            shutil.rmtree(sub)
        os.symlink(target, sub)
    old = '/content/drive/MyDrive/omni_voices'  # folder used by an earlier version
    if os.path.isdir(old):
        for f in os.listdir(old):
            if not os.path.exists(f'{base}/voices/{f}'):
                shutil.copy2(f'{old}/{f}', f'{base}/voices')
    print('✅ Saving to Google Drive: My Drive > Omni (voices + output)')
else:
    print('⚠️ Not saving to Google Drive: voices are lost when Colab closes')

In [ ]:
# 3. Launch the app: click the gradio.live link below
!python -m omni_tts.app --share

### Command-line alternative
Upload a script file, then:
```
!python -m omni_tts narrate my_script.txt --voice documentary_male
```
Output goes to `output/` (WAV + MP3).